<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/15_Transformers/01_Tokenization_and_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 15: Transfer Rumor Reliability with Transformers

**Domain:** Football (Soccer) Natural Language Processing (NLP)
**Objective:** Fine-tune a Transformer model to classify football transfer headlines as either "Credible Journalism" or "Clickbait" based entirely on linguistic patterns.

## 1. Project Context
Sports journalism, particularly during transfer windows, is saturated with highly specific vocabulary and emotional triggers. This project bypasses standard tutorial datasets to tackle a unique domain challenge: distinguishing between factual, sourced reporting (e.g., "agreement reached", "medical scheduled") and speculative clickbait (e.g., "shock swoop", "could be tempted").

## 2. Theoretical Background: Why Transformers?
Historically, deep learning for text relied on Recurrent Neural Networks (RNNs) and LSTMs. These architectures processed text sequentially (word-by-word), which was slow and prone to losing context over long sentences.

In 2017, the **Transformer** architecture revolutionized NLP by introducing the **Self-Attention Mechanism**.
* **Parallel Processing:** Transformers analyze the entire sentence simultaneously rather than sequentially.
* **Self-Attention:** The model mathematically calculates how strongly every single word in a sentence relates to every other word. For instance, in the phrase *"Chelsea launch shock £80m swoop"*, the model explicitly maps the strong contextual correlation between "shock", "swoop", and low reliability.

## 3. Project Roadmap
We will build this pipeline step-by-step to understand the mechanics beneath the code:
1. **Data Curation:** Establishing a domain-specific dataset of transfer headlines.
2. **Tokenization & Embeddings:** Translating raw text into the dense numerical matrices the model requires.
3. **Self-Attention Exploration:** Understanding how the model weighs word importance.
4. **Fine-Tuning DistilBERT:** Adapting a lightweight, pre-trained Transformer to our specific football vocabulary.
5. **Inference & Evaluation:** Testing the model against unseen transfer rumors.

In [1]:
import pandas as pd

# Creating a curated dataset of transfer rumors and headlines
data = {
    "headline": [
        "Manchester United reach full agreement with Ajax for Antony, here we go!",
        "SHOCK SWOOP: Arsenal preparing audacious £100m bid for Mbappe?",
        "Medical scheduled for tomorrow. Player will sign a five-year contract.",
        "Chelsea could be tempted to hijack Liverpool's move for midfield maestro.",
        "Official: Real Madrid announce the signing of Jude Bellingham on a six-year deal.",
        "RUMOR: Pep Guardiola 'furious' as star player demands immediate exit.",
        "Fee agreed between Tottenham and Sporting. Personal terms are not an issue.",
        "BREAKING: You won't believe which club is trying to sign Neymar!",
        "Understand Chelsea are advancing in talks to sign Caicedo. Official bid submitted.",
        "Liverpool plotting stunning double raid on Bundesliga giants."
    ],
    "label": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0] # 1 = Credible, 0 = Clickbait
}

df = pd.DataFrame(data)
df.head(10)

,headline,label
0,Manchester United reach full agreement with Aj...,1
1,SHOCK SWOOP: Arsenal preparing audacious £100m...,0
2,Medical scheduled for tomorrow. Player will si...,1
3,Chelsea could be tempted to hijack Liverpool's...,0
4,Official: Real Madrid announce the signing of ...,1
5,RUMOR: Pep Guardiola 'furious' as star player ...,0
6,Fee agreed between Tottenham and Sporting. Per...,1
7,BREAKING: You won't believe which club is tryi...,0
8,Understand Chelsea are advancing in talks to s...,1
9,Liverpool plotting stunning double raid on Bun...,0


## 4. Live Data Acquisition: Web Scraping

To train a robust model, we need real-world data. Rather than relying solely on a static CSV, we are building a live data pipeline.

We are targeting the **BBC Sport "Gossip" column**. This page is highly valuable for our specific NLP task because it aggregates daily transfer rumors from various tiers of journalism, combining credible broadsheets with sensationalist tabloids (like *The Sun* or *Daily Mail*) into a single daily feed.

**Technical Approach:**
* **`requests` & `BeautifulSoup`:** We use these libraries to fetch and parse the raw HTML of the webpage.
* **`User-Agent` Header:** Many news sites block automated Python scripts. By passing a standard web browser header, we bypass basic anti-bot protections.
* **Keyword Filtering:** We filter the scraped paragraph tags for transfer-specific vocabulary (e.g., "bid", "swoop", "fee") to isolate relevant headlines from general site text.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.bbc.com/sport/football/gossip"
# Websites block scripts without a User-Agent header
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

scraped_headlines = []

# Aggregators usually wrap rumors in standard paragraph or heading tags
for item in soup.find_all(['p', 'h3']):
    text = item.get_text().strip()

    # Filter for reasonable length and transfer-specific vocabulary
    keywords = ['sign', 'transfer', 'move', 'bid', 'contract', 'fee', 'loan', 'winger', 'striker', 'deal', 'swoop']
    if len(text) > 40 and any(kw in text.lower() for kw in keywords):
        scraped_headlines.append(text)

# Remove duplicates
scraped_headlines = list(set(scraped_headlines))

df_live = pd.DataFrame({"headline": scraped_headlines})
print(f"Successfully scraped {len(df_live)} potential transfer headlines.")
display(df_live.head(10))

Successfully scraped 8 potential transfer headlines.


,headline
0,Bellingham agrees new Real deal - Saturday's g...
1,Arsenal eye deal for teenager Mora - Saturday'...
2,Aston Villa planning Endrick move - Monday's g...
3,Villa eye Endrick loan deal - Wednesday's gossip
4,Man Utd & Arsenal eye Croatia striker - Sunday...
5,Villa could reconsider Garnacho deal - Saturda...
6,Real seek McTominay move - Friday's gossip
7,Man Utd revisit deal for Hall - Wednesday gossip


## 5. Data Cleaning & Synthetic Augmentation

Deep learning models require volume to adjust their internal weights effectively. Since a single day's web scrape only yields a handful of headlines, we must augment our live data with a synthetic dataset.

**Pipeline Steps:**
1. **String Cleaning:** Strip out structural artifacts from the web scraper (e.g., removing the " - Monday's gossip" suffixes).
2. **Labeling:** Assign our binary classifications (`1` = Credible, `0` = Clickbait).
3. **Merging & Shuffling:** Combine the live data with a programmatic synthetic dataset and randomize the row order to prevent the model from memorizing sequences.

In [3]:
# 1. Clean the live headlines by splitting at the hyphen
df_live['headline'] = df_live['headline'].str.split(' - ').str[0]

# 2. Assign labels to the live BBC Gossip data (mostly 0 for rumors/gossip)
# If a headline contains definitive words like "agrees", we label it 1.
df_live['label'] = df_live['headline'].apply(lambda x: 1 if 'agrees' in x.lower() or 'official' in x.lower() else 0)

# 3. Generate a synthetic dataset for volume (repeating core patterns)
synthetic_data = {
    "headline": [
        "Official: Manchester City announce the signing of Erling Haaland.",
        "SHOCKING: Chelsea to launch £150m mega-swoop for Mbappe!",
        "Medical completed. Arsenal to unveil new midfielder tomorrow.",
        "Rumor: Real Madrid plotting audacious double raid on Premier League.",
        "Agreement reached between Bayern and Juventus. Fee set at €70m.",
        "Sources: Liverpool could be tempted by sensational return for Mane.",
        "Here we go! Full agreement in place for the transfer.",
        "Desperate United ready to hijack rival's £90m transfer target."
    ] * 15,  # Multiply to create a batch of 120 rows
    "label": [1, 0, 1, 0, 1, 0, 1, 0] * 15
}
df_synthetic = pd.DataFrame(synthetic_data)

# 4. Merge, Shuffle, and Reset Index
df_final = pd.concat([df_live, df_synthetic]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total dataset size: {len(df_final)} rows")
display(df_final.head(10))

Total dataset size: 128 rows


,headline,label
0,Desperate United ready to hijack rival's £90m ...,0
1,Official: Manchester City announce the signing...,1
2,Rumor: Real Madrid plotting audacious double r...,0
3,Desperate United ready to hijack rival's £90m ...,0
4,Medical completed. Arsenal to unveil new midfi...,1
5,Official: Manchester City announce the signing...,1
6,Sources: Liverpool could be tempted by sensati...,0
7,Official: Manchester City announce the signing...,1
8,SHOCKING: Chelsea to launch £150m mega-swoop f...,0
9,Medical completed. Arsenal to unveil new midfi...,1


#Explanation:
The first number (0-9): This is the standard Pandas row index.

The text string: The football transfer headline itself.

The final number (0 or 1): This is the binary classification label we assigned. A 1 signifies definitive, credible journalism (e.g., "Official", "Medical completed"), while a 0 signifies speculative clickbait or unverified rumors (e.g., "Desperate", "SHOCKING").

**Step 2: Tokenization & Embeddings**

Raw text is meaningless to a neural network; it requires dense numerical matrices. We use a pre-trained **Tokenizer** to convert our English headlines into mathematical representations.

* **Token IDs:** Words (or sub-words) are mapped to specific integers from the model's fixed vocabulary.
* **Padding & Truncation:** Neural networks require uniform matrix sizes. Short sentences are padded with zeros, and long sentences are cut off at a maximum length.
* **Attention Mask:** A binary matrix that tells the Transformer which tokens represent real words (`1`) and which are just filler padding (`0`).

In [4]:
from transformers import DistilBertTokenizer

# Load the pre-trained DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Extract a single sample headline from our dataset
sample_headline = df_final['headline'].iloc[0]

# Tokenize the string into PyTorch tensors
tokens = tokenizer(
    sample_headline,
    padding='max_length',
    max_length=20,
    truncation=True,
    return_tensors="pt" # Returns PyTorch tensors
)

print(f"Original Text: {sample_headline}\n")
print(f"Token IDs:\n{tokens['input_ids']}\n")
print(f"Attention Mask:\n{tokens['attention_mask']}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Original Text: Desperate United ready to hijack rival's £90m transfer target.

Token IDs:
tensor([[  101,  7143,  2142,  3201,  2000,  7632, 17364,  6538,  1005,  1055,
          1069, 21057,  2213,  4651,  4539,  1012,   102,     0,     0,     0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]])
